In [3]:
import torch, pandas as pd
from torch.utils.data import Dataset, DataLoader
from torch import Tensor, nn
from transformers import AutoTokenizer, RobertaModel


class RoBERTaForVAD(nn.Module):
    """RoBERTa-large with a regression head for VAD prediction."""

    def __init__(self, base_model_name: str) -> None:
        super().__init__()
        self.base_model = RobertaModel.from_pretrained(base_model_name)
        self.dropout    = nn.Dropout(p=0.1)
        self.regressor  = nn.Linear(self.base_model.config.hidden_size, 3)

    def forward(self, input_ids: Tensor, attention_mask: Tensor) -> Tensor:
        outputs    = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        cls        = self.dropout(outputs.last_hidden_state[:, 0, :])
        return self.regressor(cls)



In [5]:
  # paste RoBERTaForVAD from your scorer cell here
  # Dataset
  class EmoBankDataset(Dataset):
      def __init__(self, df, tokenizer, max_length=256):
          self.encodings = tokenizer(df["text"].tolist(), truncation=True,
                                     padding=True, max_length=max_length,
                                     return_tensors="pt")
          self.labels = torch.tensor(df[["V","A","D"]].values, dtype=torch.float32)
      def __len__(self): return len(self.labels)
      def __getitem__(self, i):
          return {k: v[i] for k, v in self.encodings.items()}, self.labels[i]

  df = pd.read_csv("emobank.csv")   # columns: id, split, V, A, D, text
  train_df = df[df["split"] == "train"]
  val_df   = df[df["split"] == "dev"]
  tokenizer = AutoTokenizer.from_pretrained("roberta-large")
  train_loader = DataLoader(EmoBankDataset(train_df, tokenizer), batch_size=16, shuffle=True)

  # Detect if CUDA is available, else use CPU
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  print(f"Using device: {device}")

  model = RoBERTaForVAD("roberta-large").to(device)
  optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
  loss_fn = torch.nn.MSELoss()
  for epoch in range(3):
      model.train()
      for batch_inputs, batch_labels in train_loader:
          optimizer.zero_grad()
          preds = model(batch_inputs["input_ids"].to(device),
                        batch_inputs["attention_mask"].to(device))
          loss = loss_fn(preds, batch_labels.to(device))
          loss.backward()
          optimizer.step()
      print(f"Epoch {epoch+1} done")


  # Save
  torch.save({"model_state_dict": model.state_dict()}, "pert_emopar_weights.pt")

FileNotFoundError: [Errno 2] No such file or directory: 'emobank.csv'